# SQL Table Viewer
Connects to the SQL Server database defined in `.env` and displays the contents of a chosen table as a styled DataFrame.

In [21]:
import pyodbc
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

SQL_CONN_STR = (
    "Driver={ODBC Driver 17 for SQL Server};"
    f"Server={os.getenv('SQL_SERVER')};"
    f"Database={os.getenv('SQL_DATABASE')};"
    f"Uid={os.getenv('SQL_USERNAME')};"
    f"Pwd={{{os.getenv('SQL_PASSWORD')}}};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

print(f"Server  : {os.getenv('SQL_SERVER')}")
print(f"Database: {os.getenv('SQL_DATABASE')}")

Server  : SE-S-RA00054,1433
Database: Stressometer Reference Database


In [22]:
# ---- Configure here ----
TABLE_NAME = "order"
MAX_ROWS   = 20        # set to None for all rows
# ------------------------

In [23]:
conn   = pyodbc.connect(SQL_CONN_STR, timeout=10)
cursor = conn.cursor()

# Verify table exists
cursor.execute(
    "SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = ?",
    TABLE_NAME,
)
if cursor.fetchone()[0] == 0:
    print(f"✗ Table '{TABLE_NAME}' not found.")
    conn.close()
else:
    top_clause = f"TOP {MAX_ROWS} " if MAX_ROWS is not None else ""
    query      = f"SELECT {top_clause}* FROM dbo.[{TABLE_NAME}]"
    df         = pd.read_sql(query, conn)
    conn.close()
    print(f"✓  {len(df)} row(s) × {len(df.columns)} column(s)" +
          (f"  (capped at {MAX_ROWS})" if MAX_ROWS else ""))

✓  20 row(s) × 42 column(s)  (capped at 20)


C:\Users\SEYUYAN1\AppData\Local\Temp\ipykernel_37908\3134213625.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df         = pd.read_sql(query, conn)


In [24]:
from IPython.display import display

styled = (
    df.style
    .set_table_styles([
        {"selector": "thead th",
         "props": [("background-color", "#A6E5C6"), ("color", "white"),
                   ("font-weight", "bold"), ("text-align", "center")]},
        {"selector": "tbody tr:nth-child(even) td",
         "props": [("background-color", "#BDD485")]},
        {"selector": "td",
         "props": [("text-align", "left"), ("padding", "6px 12px")]},
    ])
    .set_caption(f"Table: {TABLE_NAME}" + (f" — top {MAX_ROWS} rows" if MAX_ROWS else ""))
    .hide(axis="index")
)

display(styled)

OrderID,OrderNo,OrginalOrderNo,CustomerID,OriginalCustomerID,BasedOnAppRelID,Contractor,DeliveryWeek,FlatnessLoggerTypeID,ElectricalSupplierID,MillBuilderID,MillName,MillType1ID,MillType2ID,MillType3ID,MSSCommID,NoOfRolls,NoOfSystems,OrderYear,ClientInterfaceID,HMIID,OtherContractor,SWReleaseID,System1ID,System2ID,HideInRefList,HidePartInRefList,BendingSystem,CoolingSystem,FlatnessLogger,MSS,MST,TSS,MTG,Y2K,History,Notes,DocumentID,TechnicalCoordinatorID,ProjectManagerID,ExportToLIAD,SSMA_TimeStamp
3,L0961.1007,None,1,273,0,1.000000,199043,None,1.000000,43,None,2,4,1,0.000000,1,1,1990,1,None,None,3,4,2,False,False,False,False,False,False,True,False,False,False,None,None,None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x8c'
4,L2961.1020,None,399,350,8,1.000000,199312,None,8.000000,11,None,2,1,1,0.000000,1,1,1992,1,None,None,0,7,2,False,False,False,False,False,False,False,False,False,False,None,None,None,13,4,True,b'\x00\x00\x00\x00\x00\x00\x18\x8d'
6,L0759.1012,None,2,2,8,4.000000,None,None,nan,21,None,8,1,2,nan,1,1,1970,3,None,None,10,1,1,False,False,False,False,False,False,False,False,False,False,None,None,None,13,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x8e'
7,L1759.1005b,None,340,3,0,1.000000,None,None,0.000000,0,None,2,5,1,0.000000,1,1,1981,3,None,None,0,2,2,False,False,False,False,False,False,False,False,False,False,None,None,None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x8f'
8,L1759.1012,None,339,4,8,1.000000,None,None,0.000000,0,None,2,7,1,0.000000,1,1,1981,3,None,None,0,2,1,False,False,False,False,False,False,False,False,False,False,None,"The roll is being re-used, see order L1961.1026.",None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x90'
9,L5695.1014,None,340,3,8,1.000000,None,None,0.000000,0,None,2,7,1,0.000000,1,1,1985,3,None,None,0,2,1,False,False,False,False,False,False,False,False,False,False,None,"Mill closed and moved, roll is used as spare for L1759.1005b/1000587.",None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x91'
11,L8695.1014,None,6,6,8,1.000000,None,None,3.000000,0,None,2,4,1,0.000000,1,1,1988,3,None,None,0,2,1,True,False,False,False,False,False,False,False,False,False,None,Replaced by BFI order 1995,None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x92'
12,L8695.1013,None,6,6,8,1.000000,None,None,3.000000,0,KW2,2,7,1,0.000000,1,1,1988,3,None,None,0,2,1,False,False,False,False,False,False,False,False,False,False,None,None,None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x93'
13,L3695.1009,L1759.1005b,340,3,0,1.000000,None,None,0.000000,0,None,2,5,0,0.000000,1,0,1983,0,None,None,0,14,2,True,False,False,False,False,False,False,False,False,False,None,Spare roll.,None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x94'
14,L5695.1011,L0759.1012,340,3,0,1.000000,None,None,0.000000,0,None,8,1,2,0.000000,1,0,1985,0,None,None,0,14,1,True,False,False,False,False,False,False,False,False,False,None,None,None,0,0,True,b'\x00\x00\x00\x00\x00\x00\x18\x95'
